# CSE457 - XỬ LÝ ÂM THANH VÀ TIẾNG NÓI
## LAB 2: ĐẶC TRƯNG TIẾNG NÓI VÀ NHẬN DẠNG BẰNG DTW
**Họ và tên:** Lưu Minh Quân
**MSSV:** 2351260684
**Lớp:** CSE457

---
### Mục tiêu
Xây dựng một hệ nhận dạng từ đơn (isolated word recognition) chạy được từ đầu đến cuối:

`Audio WAV → Endpoint detection → Frame + Hamming → MFCC → DTW với templates → Từ được nhận dạng`

Các thành phần cốt lõi (**framing, energy/ZCR, endpoint detection, autocorrelation, MFCC, khoảng cách Euclid, DTW, recognizer**) đều **tự cài đặt bằng NumPy/SciPy**, không gọi API nhận dạng có sẵn. `librosa` chỉ được dùng (nếu đã cài) để **đối chiếu** MFCC tự cài.

| Ký hiệu thư mục (ASCII) | Từ tiếng Việt |
|---|---|
| `khong` | không |
| `mot` | một |
| `hai` | hai |
| `ba` | ba |
| `bon` | bốn |

**Cấu trúc thư mục** (tương đối so với notebook):
```
lab02/
  lab02_2351260684.ipynb
  dataset/khong/khong_01.wav ... bon/bon_05.wav   # dữ liệu ghi âm (WAV mono 16 kHz)
  dataset_speaker2/...                              # (tuỳ chọn) người nói thứ 2 cho E4
  trimmed/   figures/   results.csv                 # đầu ra do notebook sinh ra
```

## 0. Cài đặt môi trường và cấu hình
Chạy cell dưới **một lần** nếu venv chưa có đủ thư viện (venv hiện có `numpy`, `soundfile`, `sounddevice`). `librosa` là tuỳ chọn.

In [1]:
# Bỏ comment dòng dưới để cài thư viện vào kernel hiện tại (chỉ cần chạy 1 lần)
%pip install numpy scipy matplotlib soundfile sounddevice scikit-learn pandas librosa

  Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached matplotlib-3.11.2-cp313-cp313-win_amd64.whl.metadata (80 kB)
  Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached pandas-3.0.6-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached librosa-1.0.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached contourpy-1.4.0-cp313-cp313-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached decorator-5.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pooch-1.


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import lfilter, resample_poly
from math import gcd
from sklearn.metrics import confusion_matrix, accuracy_score
from IPython.display import display

try:
    import soundfile as sf
except ImportError:          # fallback nếu chưa cài soundfile
    sf = None
    from scipy.io import wavfile

try:
    import librosa            # chỉ dùng để đối chiếu, không bắt buộc
except ImportError:
    librosa = None

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# ---------------- Cấu hình baseline (Mục 5.2 của đề) ----------------
FS        = 16000          # Hz
FRAME_MS  = 25             # ms
HOP_MS    = 10             # ms
WIN       = int(round(FS * FRAME_MS / 1000))   # L = 400 mẫu
HOP       = int(round(FS * HOP_MS / 1000))     # R = 160 mẫu
ALPHA     = 0.97           # pre-emphasis
NFFT      = 512
N_MELS    = 24
N_MFCC    = 13
N_TRAIN   = 3              # 3 file đầu/từ làm template, phần còn lại làm test
TOP_DB    = 35             # ngưỡng energy tương đối cho endpoint (dB dưới đỉnh)
MARGIN_MS = 50             # vùng đệm trước/sau biên endpoint
EPS       = 1e-10

LABELS = ["khong", "mot", "hai", "ba", "bon"]
VN     = {"khong": "không", "mot": "một", "hai": "hai", "ba": "ba", "bon": "bốn"}

DATA_DIR  = Path("dataset")
DATA_DIR2 = Path("dataset_speaker2")     # tuỳ chọn cho thí nghiệm E4
TRIM_DIR  = Path("trimmed")
FIG_DIR   = Path("figures")
for d in (DATA_DIR, TRIM_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Fs={FS} Hz | frame={FRAME_MS} ms -> L={WIN} mẫu | hop={HOP_MS} ms -> R={HOP} mẫu "
      f"| chồng lấn={WIN-HOP} mẫu ({(WIN-HOP)/FS*1000:.0f} ms)")
print("librosa:", "có (dùng để đối chiếu)" if librosa else "không có (không bắt buộc)")

Fs=16000 Hz | frame=25 ms -> L=400 mẫu | hop=10 ms -> R=160 mẫu | chồng lấn=240 mẫu (15 ms)
librosa: có (dùng để đối chiếu)


---
## A. Thu dữ liệu và kiểm tra chất lượng

### A.1. Ghi âm bộ dữ liệu (chạy một lần)
Đặt `RECORD = True` rồi chạy cell để ghi **5 từ × 5 lần** bằng micro (mỗi lần 1.5 s, WAV mono 16 kHz PCM-16).
Quy ước ghi âm: nói tách rời, **để 0.2–0.5 s im lặng trước và sau từ**, giữ khoảng cách micro và âm lượng ổn định.
Có thể đặt `ONLY = ["hai"]` để ghi lại riêng một từ. Nếu đã có dữ liệu thì để `RECORD = False`.

In [4]:
RECORD     = False          # <-- đổi thành True để ghi âm
N_REPEAT   = 5
REC_SEC    = 1.5
ONLY       = None           # ví dụ ["hai", "ba"] để chỉ ghi lại vài từ
SPEAKER_DIR = DATA_DIR      # đổi thành DATA_DIR2 khi ghi cho người nói thứ 2

def record_utterance(path, seconds=REC_SEC, fs=FS):
    import sounddevice as sd
    audio = sd.rec(int(seconds * fs), samplerate=fs, channels=1, dtype="float32")
    sd.wait()
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, audio[:, 0], fs, subtype="PCM_16")
    return audio[:, 0]

if RECORD:
    for lab in (ONLY or LABELS):
        for k in range(1, N_REPEAT + 1):
            p = SPEAKER_DIR / lab / f"{lab}_{k:02d}.wav"
            input(f"[{lab} = '{VN.get(lab, lab)}'] lần {k}/{N_REPEAT} - nhấn Enter rồi đợi ~0.3 s mới nói...")
            a = record_utterance(p)
            print(f"   đã lưu {p}  peak={np.max(np.abs(a)):.3f}")
else:
    print("RECORD = False -> bỏ qua ghi âm, dùng dữ liệu có sẵn trong", DATA_DIR.resolve())

RECORD = False -> bỏ qua ghi âm, dùng dữ liệu có sẵn trong D:\QUAN\code stuff\audioProcess\lab\lab02\dataset


### A.2. Đọc dữ liệu, đưa về WAV mono 16 kHz
`load_audio` đọc file, trộn về mono, resample về 16 kHz (polyphase, `resample_poly`), lọc bỏ DC offset bằng bộ lọc DC-blocking $y[n] = x[n] - x[n-1] + 0.995\,y[n-1]$ (không trừ trung bình toàn cục vì trung bình của đoạn nói sẽ làm lệch mức của silence) và chuẩn hoá đỉnh về 1.
Cell dưới cũng **ghi đè file trên đĩa thành mono 16 kHz PCM-16** nếu file chưa đúng chuẩn (yêu cầu A.2).

In [6]:
def read_wav(path):
    """Đọc WAV -> (float32 mono, sr) không resample."""
    if sf is not None:
        y, sr = sf.read(str(path), dtype="float32", always_2d=True)
        y = y.mean(axis=1)
    else:
        sr, y = wavfile.read(str(path))
        if y.dtype.kind == "i":
            y = y.astype(np.float32) / np.iinfo(y.dtype).max
        y = y.astype(np.float32)
        if y.ndim == 2:
            y = y.mean(axis=1)
    return y, sr

def to_16k(y, sr, target=FS):
    if sr == target:
        return y
    g = gcd(int(sr), int(target))
    return resample_poly(y, target // g, int(sr) // g).astype(np.float32)

def load_audio(path, normalize=True):
    y, sr = read_wav(path)
    y = to_16k(y, sr)
    y = lfilter([1, -1], [1, -0.995], y).astype(np.float32)   # DC-blocking filter (bỏ DC offset)
    if normalize:
        y = y / (np.max(np.abs(y)) + 1e-9)
    return y

def list_files(root, labels=LABELS):
    root = Path(root)
    return {lab: sorted((root / lab).glob("*.wav")) for lab in labels if (root / lab).is_dir()}

def ensure_mono16k(root):
    changed = 0
    for lab, files in list_files(root).items():
        for f in files:
            if sf is None:
                break
            info = sf.info(str(f))
            if info.samplerate != FS or info.channels != 1 or info.subtype != "PCM_16":
                y, sr = read_wav(f)
                sf.write(str(f), to_16k(y, sr), FS, subtype="PCM_16")
                changed += 1
    return changed

files_by_label = list_files(DATA_DIR)
LABELS = [lab for lab in LABELS if lab in files_by_label and len(files_by_label[lab]) > N_TRAIN]
assert len(LABELS) >= 2, ("Chưa có dữ liệu! Hãy ghi âm ở cell A.1 (RECORD=True) hoặc chép WAV vào "
                          "dataset/<nhan>/<nhan>_01.wav ...")
print("Số file được chuyển về mono 16 kHz PCM-16:", ensure_mono16k(DATA_DIR))
print("Vocabulary:", [f"{l} ({VN.get(l, l)})" for l in LABELS])

AssertionError: Chưa có dữ liệu! Hãy ghi âm ở cell A.1 (RECORD=True) hoặc chép WAV vào dataset/<nhan>/<nhan>_01.wav ...

### A.3. Bảng kiểm tra chất lượng: sampling rate, độ dài, clipping, silence đầu/cuối
- **Clipping:** tỉ lệ mẫu có |x| ≥ 0.99 trên tín hiệu *chưa chuẩn hoá* (gần full-scale ⇒ nguy cơ cắt đỉnh).
- **Silence đầu/cuối:** ước lượng nhanh bằng khoảng thời gian trước frame đầu tiên / sau frame cuối cùng có RMS > 10% RMS cực đại.

In [ ]:
def quick_silence(y, thr_ratio=0.1):
    n = 1 + max(0, (len(y) - WIN) // HOP)
    rms = np.array([np.sqrt(np.mean(y[i*HOP:i*HOP+WIN]**2)) for i in range(n)])
    idx = np.where(rms > thr_ratio * rms.max())[0]
    lead = idx[0] * HOP / FS
    trail = (len(y) - (idx[-1] * HOP + WIN)) / FS
    return lead, max(trail, 0.0)

rows = []
for lab, files in files_by_label.items():
    if lab not in LABELS:
        continue
    for f in files:
        raw, sr = read_wav(f)
        lead, trail = quick_silence(raw)
        rows.append(dict(file=f.name, label=lab, sr=sr, duration_s=round(len(raw)/sr, 3),
                         peak=round(float(np.max(np.abs(raw))), 3),
                         clip_ratio_pct=round(100*np.mean(np.abs(raw) >= 0.99), 3),
                         lead_sil_s=round(lead, 3), trail_sil_s=round(trail, 3)))
qc = pd.DataFrame(rows)
display(qc)
print(f"Tổng số file: {len(qc)} | sr duy nhất: {sorted(qc.sr.unique())} | "
      f"file có clipping (>0.1% mẫu): {int((qc.clip_ratio_pct > 0.1).sum())} | "
      f"file silence đầu hoặc cuối < 0.1 s: {int(((qc.lead_sil_s < 0.1) | (qc.trail_sil_s < 0.1)).sum())}")

In [ ]:
# Waveform của (ít nhất) 3 từ khác nhau - lần ghi đầu tiên
show = LABELS[:max(3, min(5, len(LABELS)))]
fig, axes = plt.subplots(len(show), 1, figsize=(10, 1.9*len(show)), sharex=True)
for ax, lab in zip(np.atleast_1d(axes), show):
    y = load_audio(files_by_label[lab][0], normalize=False)
    t = np.arange(len(y)) / FS
    ax.plot(t, y, lw=0.6, color="#3b6998")
    ax.axhline(0.99, color="r", ls=":", lw=0.8); ax.axhline(-0.99, color="r", ls=":", lw=0.8)
    ax.set_ylabel("Biên độ"); ax.set_title(f"'{VN.get(lab, lab)}' ({files_by_label[lab][0].name})", fontsize=10)
np.atleast_1d(axes)[-1].set_xlabel("Thời gian (s)")
fig.suptitle("A. Waveform các từ (đường đỏ: ngưỡng clipping ±0.99)", y=1.0)
fig.tight_layout(); fig.savefig(FIG_DIR / "A_waveforms.png", bbox_inches="tight"); plt.show()

### A.4. Chia train/test **trước** khi chọn template
Với mỗi từ: `N_TRAIN = 3` file đầu (theo thứ tự tên) → **template (train)**, các file còn lại → **test**.
Kiểm tra bằng `assert` rằng không có file nào vừa ở train vừa ở test (tránh data leakage).

In [ ]:
train_files = {lab: files_by_label[lab][:N_TRAIN] for lab in LABELS}
test_files  = {lab: files_by_label[lab][N_TRAIN:] for lab in LABELS}
all_train = {str(f) for v in train_files.values() for f in v}
all_test  = {str(f) for v in test_files.values() for f in v}
assert all_train.isdisjoint(all_test), "Data leakage: có file nằm ở cả train và test!"
print(pd.DataFrame({"train (template)": {l: len(v) for l, v in train_files.items()},
                    "test": {l: len(v) for l, v in test_files.items()}}).T.to_string())
print(f"\nTổng: {len(all_train)} template, {len(all_test)} file test - không trùng lặp ✔")

**Nhận xét (A):** Mọi file được đưa về cùng chuẩn mono/16 kHz trước khi trích đặc trưng (tránh lỗi "không cùng sampling rate" ở Mục 9).
Cột `peak`/`clip_ratio_pct` cho biết có bị cắt đỉnh hay không; cột `lead_sil_s`/`trail_sil_s` cho biết có đủ 0.2–0.5 s silence cho endpoint detection không — file nào silence < 0.1 s hoặc clipping cần ghi lại.
Việc chia train/test được cố định theo tên file trước mọi bước chọn template nên không có rò rỉ dữ liệu.

---
## B. Đặc trưng miền thời gian: energy, magnitude, RMS, ZCR, autocorrelation

Chia frame: $L = \mathrm{round}(F_s T_f) = 400$, $R = \mathrm{round}(F_s T_h) = 160$; cửa sổ Hamming $w[n] = 0.54 - 0.46\cos\left(\frac{2\pi n}{L-1}\right)$.

$$E_r = \sum_n x_r^2[n],\quad M_r = \sum_n |x_r[n]|,\quad \mathrm{RMS}_r = \sqrt{\tfrac{1}{L}\sum_n x_r^2[n]},\quad E_r(\mathrm{dB}) = 10\log_{10}(E_r+\varepsilon)$$

$$Z_r = \frac{1}{2L}\sum_m \left|\mathrm{sgn}(x[m]) - \mathrm{sgn}(x[m-1])\right|\quad(\text{cửa sổ chữ nhật})$$

In [ ]:
def frame_signal(x, win=WIN, hop=HOP):
    """Chia tín hiệu thành ma trận (n_frames, win); pad 0 ở cuối để không mất mẫu."""
    if len(x) < win:
        x = np.pad(x, (0, win - len(x)))
    n = 1 + int(np.ceil((len(x) - win) / hop))
    x = np.pad(x, (0, (n - 1) * hop + win - len(x)))
    idx = np.arange(win)[None, :] + hop * np.arange(n)[:, None]
    return x[idx]

def hamming(L):
    n = np.arange(L)
    return 0.54 - 0.46 * np.cos(2 * np.pi * n / (L - 1))

def short_time_features(y):
    fr = frame_signal(y)
    xw = fr * hamming(WIN)
    E   = np.sum(xw**2, axis=1)
    M   = np.sum(np.abs(xw), axis=1)
    RMS = np.sqrt(np.mean(xw**2, axis=1))
    EdB = 10 * np.log10(E + EPS)
    sgn = np.where(fr >= 0, 1, -1)                       # sgn(x)=1 nếu x>=0
    ZCR = np.sum(np.abs(np.diff(sgn, axis=1)), axis=1) / (2 * WIN)
    t   = (np.arange(len(fr)) * HOP + WIN / 2) / FS      # thời điểm tâm frame
    return dict(t=t, E=E, M=M, RMS=RMS, EdB=EdB, ZCR=ZCR)

# Kiểm tra nhanh với ví dụ trong đề: frame 100 mẫu, 20 lần đổi dấu -> ZCR = 0.20
_x = np.tile([1.0]*5 + [-1.0]*5, 10)[:100]
_s = np.where(_x >= 0, 1, -1)
print("ZCR ví dụ đề =", np.sum(np.abs(np.diff(_s))) / (2 * 100), "(19 lần đổi dấu trong 100 mẫu ≈ 0.19-0.20)")
print("Ví dụ RMS 0.20 vs 0.02 ->", 10*np.log10((0.20/0.02)**2), "dB")

In [ ]:
def plot_time_features(lab, path, fname):
    y = load_audio(path)
    F = short_time_features(y)
    t = np.arange(len(y)) / FS
    fig, ax = plt.subplots(4, 1, figsize=(10, 7.5), sharex=True)
    ax[0].plot(t, y, lw=0.5, color="#3b6998"); ax[0].set_ylabel("Biên độ")
    ax[0].set_title(f"B. '{VN.get(lab, lab)}' - {path.name}: waveform, energy, RMS và ZCR (frame {FRAME_MS} ms, hop {HOP_MS} ms)")
    ax[1].plot(F["t"], F["EdB"], color="#e15759"); ax[1].set_ylabel("Log-energy (dB)")
    ax[2].plot(F["t"], F["RMS"], color="#59a14f", label="RMS")
    ax[2].plot(F["t"], F["M"] / WIN, color="#b07aa1", ls="--", label="Magnitude/L")
    ax[2].set_ylabel("Mức biên độ"); ax[2].legend(fontsize=8, loc="upper right")
    ax[3].plot(F["t"], F["ZCR"], color="#f28e2b"); ax[3].set_ylabel("ZCR (crossing/mẫu)")
    ax[3].set_xlabel("Thời gian (s)")
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, bbox_inches="tight"); plt.show()
    return F

feat_examples = {}
for lab in LABELS[:3]:
    feat_examples[lab] = plot_time_features(lab, files_by_label[lab][0], f"B_energy_zcr_{lab}.png")

### B.2. Thống kê silence / voiced / unvoiced
Phân loại thô từng frame (chỉ để minh hoạ, không dùng cho nhận dạng):
- **silence**: log-energy < (đỉnh − 35 dB)
- **voiced**: energy cao **và** ZCR thấp (< 0.15)
- **unvoiced**: có năng lượng nhưng ZCR cao (≥ 0.15) — ví dụ phụ âm xát /x/ trong "không", /h/ trong "hai".

In [ ]:
def classify_frames(F, top_db=TOP_DB, zcr_thr=0.15):
    speech = F["EdB"] > F["EdB"].max() - top_db
    cls = np.where(~speech, "silence", np.where(F["ZCR"] < zcr_thr, "voiced", "unvoiced"))
    return cls

rows = []
for lab, F in feat_examples.items():
    cls = classify_frames(F)
    for c in ["silence", "voiced", "unvoiced"]:
        m = cls == c
        rows.append(dict(word=VN.get(lab, lab), cls=c, n_frames=int(m.sum()),
                         mean_EdB=round(float(F["EdB"][m].mean()), 1) if m.any() else np.nan,
                         mean_ZCR=round(float(F["ZCR"][m].mean()), 3) if m.any() else np.nan))
display(pd.DataFrame(rows).pivot(index="word", columns="cls", values=["n_frames", "mean_EdB", "mean_ZCR"]))

### B.3. Short-time autocorrelation và ước lượng pitch
$R_r[k] = \sum_n x_r[n]\,x_r[n+k]$. Với frame hữu thanh, đỉnh đầu tiên sau $k=0$ nằm gần chu kỳ pitch $N_0$ ⇒ $F_0 \approx F_s/N_0$.
Tìm đỉnh trong dải lag tương ứng 60–400 Hz. Frame vô thanh / silence không có đỉnh rõ (tỉ số $R[N_0]/R[0]$ thấp).

In [ ]:
def autocorr(frame):
    N = len(frame)
    return np.array([np.sum(frame[:N-k] * frame[k:]) for k in range(N)])

def pitch_autocorr(frame, fmin=60, fmax=400):
    R = autocorr(frame * hamming(len(frame)))
    kmin, kmax = int(FS / fmax), int(FS / fmin)
    k0 = kmin + np.argmax(R[kmin:kmax])
    return FS / k0, k0, R[k0] / (R[0] + EPS), R

lab0 = LABELS[0]
y0 = load_audio(files_by_label[lab0][0]); F0f = short_time_features(y0)
cls0 = classify_frames(F0f)
fr0 = frame_signal(y0)
picks = {}
for c in ["voiced", "unvoiced", "silence"]:
    idx = np.where(cls0 == c)[0]
    if len(idx):
        picks[c] = idx[np.argmax(F0f["EdB"][idx])] if c != "silence" else idx[0]

fig, axes = plt.subplots(len(picks), 2, figsize=(11, 2.6*len(picks)))
axes = np.atleast_2d(axes)
for row, (c, i) in enumerate(picks.items()):
    f0, k0, ratio, R = pitch_autocorr(fr0[i])
    tt = np.arange(WIN) / FS * 1000
    axes[row, 0].plot(tt, fr0[i], lw=0.8); axes[row, 0].set_title(f"Frame {i} ({c}) - '{VN.get(lab0, lab0)}'", fontsize=10)
    axes[row, 0].set_xlabel("ms")
    axes[row, 1].plot(np.arange(WIN), R / (R[0] + EPS), lw=0.8)
    axes[row, 1].axvline(k0, color="r", ls="--", lw=0.8)
    axes[row, 1].set_title(f"Autocorrelation chuẩn hoá: N0={k0}, F0≈{f0:.0f} Hz, R[N0]/R[0]={ratio:.2f}", fontsize=10)
    axes[row, 1].set_xlabel("lag k (mẫu)")
fig.tight_layout(); fig.savefig(FIG_DIR / "B_autocorrelation.png", bbox_inches="tight"); plt.show()

# Đường pitch theo thời gian (chỉ tính ở frame voiced)
f0_track = np.array([pitch_autocorr(fr0[i])[0] if cls0[i] == "voiced" else np.nan for i in range(len(fr0))])
print(f"F0 trung bình các frame voiced của '{VN.get(lab0, lab0)}': {np.nanmean(f0_track):.1f} Hz "
      f"(min {np.nanmin(f0_track):.0f}, max {np.nanmax(f0_track):.0f})")

**Nhận xét (B):**
- **Silence:** log-energy thấp, gần mức nền (thấp hơn đỉnh hàng chục dB); ZCR của silence phụ thuộc nhiễu nền — nhiễu trắng nhỏ cho ZCR cao nhưng năng lượng rất thấp, silence "số" (toàn 0) cho ZCR ≈ 0.
- **Voiced (nguyên âm a, ô, ôn, ai…):** năng lượng cao nhất, ZCR thấp (≈ 0.02–0.1) do năng lượng tập trung ở tần số thấp (F0 và formant đầu); autocorrelation có đỉnh tuần hoàn rõ ở lag $N_0$, cho $F_0$ hợp lý (≈ 100–150 Hz với giọng nam, 180–250 Hz giọng nữ).
- **Unvoiced (phụ âm xát /x/ của "không", /h/ của "hai"):** năng lượng thấp hơn nguyên âm nhưng ZCR cao (> 0.15–0.3) vì phổ giống nhiễu tần số cao; autocorrelation giảm nhanh, không có đỉnh tuần hoàn ($R[N_0]/R[0]$ nhỏ).
- Energy (bình phương) có dynamic range lớn hơn magnitude, nên làm nổi rõ vùng nguyên âm; magnitude/RMS mượt hơn.

---
## C. Endpoint detection

**Thuật toán tự cài** (theo ý tưởng Rabiner–Schafer: log-energy + tinh chỉnh bằng ZCR):
1. Tính log-energy $E(\mathrm{dB})$ và ZCR cho từng frame.
2. Ước lượng **mức nền** = phân vị 10% của $E(\mathrm{dB})$ (các frame yên lặng nhất); ZCR nền = trung bình ZCR của các frame có $E < $ nền + 3 dB.
3. Ngưỡng energy: $T_E = \max(E_{\max} - \text{TOP\_DB},\; \text{nền} + 10\,\mathrm{dB})$ với `TOP_DB = 35` dB. Vùng speech thô = từ frame đầu tiên đến frame cuối cùng có $E > T_E$ (bỏ các "click" ngắn < 3 frame liên tiếp).
4. **Tinh chỉnh bằng ZCR:** nhìn lùi tối đa 250 ms trước điểm đầu (và tiến 250 ms sau điểm cuối). Nếu có ≥ 3 frame với ZCR > $T_Z = \max(0.25,\ \mu_Z + 2\sigma_Z)$ **và** năng lượng > nền + 3 dB thì dời biên ra frame xa nhất đó (giữ lại phụ âm vô thanh năng lượng thấp như /x/, /h/).
5. Thêm **margin 50 ms** mỗi bên.

Ngưỡng báo cáo: `TOP_DB = 35 dB`, sàn `nền + 10 dB`, $T_Z = \max(0.25, \mu_Z+2\sigma_Z)$, `MARGIN_MS = 50 ms`. Đây là heuristic của bài lab, không phải ngưỡng cố định trong giáo trình.

In [ ]:
def _runs(mask):
    """Trả về danh sách (start, end) các đoạn True liên tiếp (end bao gồm)."""
    idx = np.flatnonzero(np.diff(np.r_[0, mask.astype(int), 0]))
    return list(zip(idx[::2], idx[1::2] - 1))

def endpoint_detect(y, top_db=TOP_DB, margin_ms=MARGIN_MS, use_zcr=True, min_run=3,
                    zcr_search_ms=250, return_info=False):
    F = short_time_features(y)
    EdB, Z = F["EdB"], F["ZCR"]
    noise = np.percentile(EdB, 10)
    T_E = max(EdB.max() - top_db, noise + 10)
    sil = EdB < noise + 3
    muZ, sdZ = (Z[sil].mean(), Z[sil].std()) if sil.sum() > 2 else (0.0, 0.0)
    T_Z = max(0.25, muZ + 2 * sdZ)

    runs = [r for r in _runs(EdB > T_E) if r[1] - r[0] + 1 >= min_run] or _runs(EdB > T_E)
    s_f, e_f = runs[0][0], runs[-1][1]
    s_e, e_e = s_f, e_f                                  # biên chỉ dựa trên energy

    if use_zcr:
        K = int(zcr_search_ms / HOP_MS)
        cand = (Z > T_Z) & (EdB > noise + 3)
        lo = max(0, s_f - K); before = np.flatnonzero(cand[lo:s_f]) + lo
        if len(before) >= 3:
            s_f = before[0]
        hi = min(len(Z), e_f + 1 + K); after = np.flatnonzero(cand[e_f + 1:hi]) + e_f + 1
        if len(after) >= 3:
            e_f = after[-1]

    m = int(FS * margin_ms / 1000)
    s = max(0, s_f * HOP - m)
    e = min(len(y), e_f * HOP + WIN + m)
    if return_info:
        return y[s:e], (s, e), dict(F=F, noise=noise, T_E=T_E, T_Z=T_Z, energy_only=(s_e, e_e),
                                    frames=(s_f, e_f))
    return y[s:e], (s, e)

def trim_energy_baseline(y, top_db=TOP_DB, margin_ms=MARGIN_MS):
    """Baseline của đề (librosa.effects.trim) - tự viết lại bằng NumPy để không phụ thuộc librosa:
    giữ đoạn từ frame đầu đến frame cuối có RMS(dB) > max - top_db."""
    fr = frame_signal(y)
    rms_db = 20 * np.log10(np.sqrt(np.mean(fr**2, axis=1)) + EPS)
    nz = np.flatnonzero(rms_db > rms_db.max() - top_db)
    m = int(FS * margin_ms / 1000)
    s = max(0, nz[0] * HOP - m); e = min(len(y), nz[-1] * HOP + WIN + m)
    return y[s:e], (s, e)

In [ ]:
# Chạy endpoint cho toàn bộ dataset, xuất file đã trim, so sánh thời lượng trước/sau
rows = []
for lab in LABELS:
    for f in files_by_label[lab]:
        y = load_audio(f)
        yt, (s, e) = endpoint_detect(y)
        _, (sb, eb) = trim_energy_baseline(y)
        (TRIM_DIR / lab).mkdir(parents=True, exist_ok=True)
        if sf is not None:
            sf.write(str(TRIM_DIR / lab / f.name), yt, FS, subtype="PCM_16")
        else:
            wavfile.write(str(TRIM_DIR / lab / f.name), FS, (yt * 32767).astype(np.int16))
        rows.append(dict(file=f.name, label=lab, before_s=round(len(y)/FS, 3), after_s=round((e-s)/FS, 3),
                         start_s=round(s/FS, 3), end_s=round(e/FS, 3),
                         baseline_after_s=round((eb-sb)/FS, 3)))
ep = pd.DataFrame(rows)
ep["removed_pct"] = (100 * (1 - ep.after_s / ep.before_s)).round(1)
display(ep)
print(ep.groupby("label")[["before_s", "after_s", "baseline_after_s", "removed_pct"]].mean().round(3))
print(f"\nĐã xuất {len(ep)} file trim vào {TRIM_DIR.resolve()}")

In [ ]:
def plot_endpoint(lab, path, fname):
    y = load_audio(path)
    _, (s, e), info = endpoint_detect(y, return_info=True)
    _, (s0, e0) = endpoint_detect(y, use_zcr=False, margin_ms=0)
    F = info["F"]; t = np.arange(len(y)) / FS
    fig, ax = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True)
    ax[0].plot(t, y, lw=0.5, color="#3b6998")
    ax[0].axvspan(s/FS, e/FS, color="g", alpha=0.12, label="giữ lại (energy+ZCR+margin)")
    for v in (s0/FS, e0/FS):
        ax[0].axvline(v, color="k", ls=":", lw=1)
    ax[0].plot([], [], "k:", label="biên chỉ dùng energy, không margin")
    ax[0].legend(fontsize=8, loc="upper right"); ax[0].set_ylabel("Biên độ")
    ax[0].set_title(f"C. Endpoint detection - '{VN.get(lab, lab)}' ({path.name})")
    ax[1].plot(F["t"], F["EdB"], color="#e15759")
    ax[1].axhline(info["T_E"], color="k", ls="--", lw=0.8, label=f"T_E = {info['T_E']:.1f} dB")
    ax[1].axhline(info["noise"], color="gray", ls=":", lw=0.8, label=f"nền = {info['noise']:.1f} dB")
    ax[1].legend(fontsize=8); ax[1].set_ylabel("Log-energy (dB)")
    ax[2].plot(F["t"], F["ZCR"], color="#f28e2b")
    ax[2].axhline(info["T_Z"], color="k", ls="--", lw=0.8, label=f"T_Z = {info['T_Z']:.2f}")
    ax[2].legend(fontsize=8); ax[2].set_ylabel("ZCR"); ax[2].set_xlabel("Thời gian (s)")
    for a in ax:
        a.axvline(s/FS, color="g", lw=1); a.axvline(e/FS, color="g", lw=1)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, bbox_inches="tight"); plt.show()
    return (s, e), (s0, e0)

# "không" bắt đầu bằng phụ âm xát vô thanh /x/ (kh) năng lượng thấp; "hai" bắt đầu bằng /h/
for lab in [l for l in ["khong", "hai"] if l in LABELS] or LABELS[:1]:
    (s, e), (s0, e0) = plot_endpoint(lab, files_by_label[lab][0], f"C_endpoint_{lab}.png")
    print(f"'{VN.get(lab, lab)}': energy-only bắt đầu {s0/FS*1000:.0f} ms | energy+ZCR+margin bắt đầu "
          f"{s/FS*1000:.0f} ms -> giữ thêm {(s0-s)/FS*1000:.0f} ms ở đầu, {(e-e0)/FS*1000:.0f} ms ở cuối")

### C.2. Ảnh hưởng của margin đến phụ âm đầu/cuối
Thử margin = 0 / 20 / 50 / 100 ms trên từ có phụ âm đầu năng lượng thấp. Nghe lại các file trong `trimmed/` (cell dưới hiển thị player) để chắc chắn không mất âm /kh/, /h/ hay âm mũi cuối /n/, /ŋ/.

In [ ]:
from IPython.display import Audio
lab_c = "khong" if "khong" in LABELS else LABELS[0]
y = load_audio(files_by_label[lab_c][0])
rows = []
for mg in [0, 20, 50, 100]:
    yt, (s, e) = endpoint_detect(y, margin_ms=mg)
    yt0, (s0, e0) = endpoint_detect(y, margin_ms=mg, use_zcr=False)
    rows.append(dict(margin_ms=mg, start_ms_energy_only=round(s0/FS*1000), start_ms_with_zcr=round(s/FS*1000),
                     end_ms=round(e/FS*1000), duration_ms=round((e-s)/FS*1000)))
display(pd.DataFrame(rows))
print("Bản gốc:"); display(Audio(y, rate=FS))
print(f"Bản đã trim (margin {MARGIN_MS} ms):"); display(Audio(endpoint_detect(y)[0], rate=FS))

**Nhận xét (C):** Endpoint detection loại bỏ phần lớn silence đầu/cuối (cột `removed_pct`), giúp DTW không phải căn chỉnh các đoạn nền không liên quan.
Chỉ dùng ngưỡng energy dễ **cắt mất phụ âm vô thanh năng lượng thấp** ở đầu từ ("không" bắt đầu bằng /x/, "hai" bằng /h/): trên đồ thị, đoạn phụ âm này có log-energy dưới $T_E$ nhưng ZCR vượt $T_Z$, nên bước tinh chỉnh ZCR kéo biên đầu sớm hơn. Margin 50 ms thêm một lớp an toàn cho âm mũi cuối (/ŋ/ trong "không", /n/ trong "bốn") vốn có năng lượng giảm dần. Margin 0 ms có rủi ro cắt sát; margin quá lớn (100 ms) lại giữ thêm silence.

---
## D. MFCC (tự cài đặt)
Pipeline: **pre-emphasis → frame/Hamming → FFT/power → Mel filterbank → log → DCT → (CMN)**

- Pre-emphasis $y[n] = x[n] - 0.97\,x[n-1]$ (dùng **cùng α** cho train và test).
- $P_r[k] = |X_r[k]|^2 / N_{FFT}$, $N_{FFT}=512$.
- Thang Mel (Huang–Acero–Hon): $B(f) = 1125\ln(1 + f/700)$, $B^{-1}(b) = 700(e^{b/1125}-1)$; $M=24$ bộ lọc tam giác cách đều trên thang Mel từ 0 đến 8 kHz.
- $S_r[m] = \ln\left(\sum_k P_r[k] H_m[k] + \varepsilon\right)$; $c_r[n] = \sum_{m=0}^{M-1} S_r[m]\cos\left(\pi n (m+\tfrac12)/M\right)$, giữ $n = 0..12$.
- **CMN:** trừ trung bình theo thời gian của từng utterance (áp dụng nhất quán cho train và test).

In [ ]:
def hz2mel(f):  return 1125.0 * np.log(1.0 + np.asarray(f) / 700.0)
def mel2hz(b):  return 700.0 * (np.exp(np.asarray(b) / 1125.0) - 1.0)

def mel_filterbank(n_mels=N_MELS, nfft=NFFT, fs=FS, fmin=0.0, fmax=None):
    fmax = fmax or fs / 2
    mel_pts = np.linspace(hz2mel(fmin), hz2mel(fmax), n_mels + 2)
    hz_pts = mel2hz(mel_pts)
    freqs = np.arange(nfft // 2 + 1) * fs / nfft
    H = np.zeros((n_mels, nfft // 2 + 1))
    for m in range(n_mels):
        fl, fc, fr = hz_pts[m], hz_pts[m + 1], hz_pts[m + 2]
        up = (freqs - fl) / (fc - fl)
        down = (fr - freqs) / (fr - fc)
        H[m] = np.maximum(0, np.minimum(up, down))
    return H, hz_pts

def dct_matrix(n_mfcc=N_MFCC, M=N_MELS):
    n = np.arange(n_mfcc)[:, None]; m = np.arange(M)[None, :]
    return np.cos(np.pi * n * (m + 0.5) / M)               # (n_mfcc, M)

MEL_FB, MEL_HZ = mel_filterbank()
DCT = dct_matrix()

def mfcc_feature(y, alpha=ALPHA, cmn=True, n_mfcc=N_MFCC, return_all=False):
    y = lfilter([1.0, -alpha], [1.0], y) if alpha else y      # pre-emphasis
    fr = frame_signal(y) * hamming(WIN)                        # (T, L)
    P = np.abs(np.fft.rfft(fr, n=NFFT, axis=1))**2 / NFFT      # (T, 257)
    S = np.log(P @ MEL_FB.T + EPS)                             # (T, 24) log mel energy
    C = S @ DCT[:n_mfcc].T                                     # (T, 13)
    if cmn:
        C = C - C.mean(axis=0, keepdims=True)
    if return_all:
        return C, dict(P=P, S=S)
    return C                                                   # shape (T frames, 13): MỖI HÀNG LÀ 1 FRAME

def delta(C, N=2):
    """Hệ số động Δ theo công thức hồi quy (giống librosa.feature.delta, width=2N+1)."""
    T = len(C); pad = np.pad(C, ((N, N), (0, 0)), mode="edge")
    num = sum(n * (pad[N+n:N+n+T] - pad[N-n:N-n+T]) for n in range(1, N + 1))
    return num / (2 * sum(n * n for n in range(1, N + 1)))

def mfcc_delta(y, **kw):
    C = mfcc_feature(y, **kw)
    return np.hstack([C, delta(C)])                            # (T, 26)

In [ ]:
# Minh hoạ Mel filterbank 24 bộ lọc
freqs = np.arange(NFFT // 2 + 1) * FS / NFFT
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw=dict(width_ratios=[2.2, 1]))
for m in range(N_MELS):
    ax[0].plot(freqs, MEL_FB[m], lw=1)
ax[0].set_title(f"D. Mel filterbank {N_MELS} bộ lọc, Fs = {FS/1000:.0f} kHz, NFFT = {NFFT}")
ax[0].set_xlabel("Tần số (Hz)"); ax[0].set_ylabel("Trọng số")
bw = MEL_HZ[2:] - MEL_HZ[:-2]
ax[1].bar(np.arange(N_MELS), bw, color="#4e79a7")
ax[1].set_title("Băng thông mỗi bộ lọc"); ax[1].set_xlabel("Chỉ số bộ lọc m"); ax[1].set_ylabel("Hz")
fig.tight_layout(); fig.savefig(FIG_DIR / "D_mel_filterbank.png", bbox_inches="tight"); plt.show()
print("Tần số trung tâm (Hz):", np.round(MEL_HZ[1:-1]).astype(int))
print(f"Băng thông bộ lọc đầu = {bw[0]:.0f} Hz, bộ lọc cuối = {bw[-1]:.0f} Hz")

In [ ]:
# Các bước trung gian và ma trận MFCC (heatmap) cho >= 2 từ khác nhau (đã trim endpoint)
show = LABELS[:min(4, len(LABELS))]
fig, axes = plt.subplots(2, len(show), figsize=(3.6*len(show), 6.2))
mfcc_shapes = {}
for j, lab in enumerate(show):
    f = files_by_label[lab][0]
    yt, _ = endpoint_detect(load_audio(f))
    C, inter = mfcc_feature(yt, return_all=True)
    mfcc_shapes[f.name] = C.shape
    ext = [0, len(C) * HOP / FS, 0, None]
    im0 = axes[0, j].imshow(inter["S"].T, origin="lower", aspect="auto", cmap="magma",
                            extent=[0, len(C)*HOP/FS, 0, N_MELS])
    axes[0, j].set_title(f"log Mel energy - '{VN.get(lab, lab)}'", fontsize=10)
    axes[0, j].set_ylabel("Bộ lọc Mel m"); fig.colorbar(im0, ax=axes[0, j], fraction=0.046)
    vlim = np.percentile(np.abs(C[:, 1:]), 98)          # thang màu theo c1..c12 (c0 có biên độ lớn hơn nhiều)
    im1 = axes[1, j].imshow(C.T, origin="lower", aspect="auto", cmap="coolwarm", vmin=-vlim, vmax=vlim,
                            extent=[0, len(C)*HOP/FS, -0.5, N_MFCC-0.5])
    axes[1, j].set_title(f"MFCC (CMN) - shape {C.shape}", fontsize=10)
    for a in (axes[0, j], axes[1, j]): a.grid(False)
    axes[1, j].set_xlabel("Thời gian (s)"); axes[1, j].set_ylabel("Hệ số c[n]")
    fig.colorbar(im1, ax=axes[1, j], fraction=0.046)
fig.suptitle("D. Log Mel filterbank và MFCC heatmap của các từ khác nhau", y=1.0)
fig.tight_layout(); fig.savefig(FIG_DIR / "D_mfcc_heatmaps.png", bbox_inches="tight"); plt.show()

print("Shape MFCC (T frames, 13) của các file:")
for k, v in mfcc_shapes.items():
    print(f"  {k:<14} -> {v}")

In [ ]:
# (Tuỳ chọn) đối chiếu với librosa: chỉ để kiểm tra, không dùng cho nhận dạng
if librosa is not None:
    yt, _ = endpoint_detect(load_audio(files_by_label[LABELS[0]][0]))
    mine = mfcc_feature(yt, cmn=False)
    lib = librosa.feature.mfcc(y=lfilter([1, -ALPHA], [1], yt), sr=FS, n_mfcc=N_MFCC, n_mels=N_MELS,
                               n_fft=NFFT, win_length=WIN, hop_length=HOP, window="hamming",
                               center=False, htk=True).T
    T = min(len(mine), len(lib))
    corr = [np.corrcoef(mine[:T, i], lib[:T, i])[0, 1] for i in range(1, N_MFCC)]
    print("Tương quan theo thời gian giữa MFCC tự cài và librosa (c1..c12):", np.round(corr, 3))
    print("(librosa dùng log dB và DCT trực chuẩn nên thang đo khác, nhưng hình dạng theo thời gian gần như trùng.)")
else:
    print("Không có librosa - bỏ qua bước đối chiếu.")

**Nhận xét (D):**
- Mel filterbank dày đặc và hẹp ở tần số thấp, thưa và rộng dần ở tần số cao (băng thông bộ lọc cuối lớn gấp nhiều lần bộ lọc đầu) — mô phỏng độ phân giải tần số của tai người.
- Heatmap MFCC của các từ khác nhau có **cấu trúc theo thời gian khác nhau**: ví dụ "không" có đoạn đầu ZCR/phổ cao (phụ âm /x/) rồi chuyển sang nguyên âm tròn môi /o/ và âm mũi /ŋ/; "hai" có chuyển động formant từ /a/ sang /i/ (bán nguyên âm cuối) thể hiện thành sự thay đổi dần của c1–c4; "ba" gần như ổn định trên một nguyên âm /a/ dài sau âm tắc /b/.
- **Vì sao số frame khác nhau nhưng số chiều/frame cố định?** Số frame $T = 1 + \lceil (N - L)/R \rceil$ phụ thuộc độ dài tín hiệu $N$ (tốc độ nói, độ dài sau trim), còn mỗi frame luôn qua cùng FFT 512 điểm → cùng 24 bộ lọc Mel → cùng DCT giữ 13 hệ số. Vì vậy mỗi utterance là ma trận $(T \times 13)$ với $T$ thay đổi — chính là lý do cần DTW để so sánh.
- Tham số (α = 0.97, Hamming, n_mels = 24, n_mfcc = 13, CMN) được cố định trong một hàm duy nhất `mfcc_feature` dùng chung cho train và test.

---
## E. Dynamic Time Warping (tự cài đặt)

- Local distance Euclid: $C[i,j] = d(x_i, y_j) = \lVert x_i - y_j \rVert_2$.
- Truy hồi (3 bước cục bộ): $D[i,j] = C[i,j] + \min\{D[i-1,j],\ D[i,j-1],\ D[i-1,j-1]\}$, bắt đầu $(0,0)$, kết thúc $(N-1, M-1)$, đường đi đơn điệu.
- Backtracking từ ô cuối để lấy optimal path $P$; chuẩn hoá $\mathrm{DTW_{norm}} = D[N-1, M-1] / |P|$.

In [ ]:
def euclid(a, b):
    """Khoảng cách Euclid giữa 2 vector đặc trưng."""
    return np.sqrt(np.sum((a - b) ** 2))

def local_distance_matrix(X, Y):
    """C[i,j] = ||x_i - y_j||_2, tính bằng broadcasting (tương đương gọi euclid cho mọi cặp)."""
    diff = X[:, None, :] - Y[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=2))

print("Ví dụ đề: d([1,2],[4,6]) =", euclid(np.array([1., 2.]), np.array([4., 6.])))

def dtw(X, Y, return_path=True):
    """DTW tự cài bằng quy hoạch động.
    X: (N, D), Y: (M, D). Trả về (DTW_norm, path, C, D_acc)."""
    C = local_distance_matrix(X, Y)
    N, M = C.shape
    D = np.full((N + 1, M + 1), np.inf)
    D[0, 0] = 0.0
    back = np.zeros((N + 1, M + 1), dtype=np.int8)   # 0: chéo, 1: dọc (i-1,j), 2: ngang (i,j-1)
    for i in range(1, N + 1):
        Ci = C[i - 1]; Dp = D[i - 1]; Dc = D[i]
        for j in range(1, M + 1):
            diag, up, left = Dp[j - 1], Dp[j], Dc[j - 1]
            if diag <= up and diag <= left:
                best, b = diag, 0
            elif up <= left:
                best, b = up, 1
            else:
                best, b = left, 2
            Dc[j] = Ci[j - 1] + best
            back[i, j] = b
    # backtracking
    path = []
    i, j = N, M
    while i > 0 and j > 0:
        path.append((i - 1, j - 1))
        b = back[i, j]
        if b == 0:   i, j = i - 1, j - 1
        elif b == 1: i -= 1
        else:        j -= 1
    path.reverse()
    return D[N, M] / len(path), path, C, D[1:, 1:]

# Kiểm tra bắt buộc: X và Y giống hệt -> DTW_norm ≈ 0
Xt = mfcc_feature(endpoint_detect(load_audio(train_files[LABELS[0]][0]))[0])
d_same, p_same, _, _ = dtw(Xt, Xt)
print(f"DTW_norm(X, X) = {d_same:.2e} (|P| = {len(p_same)} = số frame {len(Xt)}, path nằm đúng đường chéo)")
# Kiểm tra với tín hiệu bị kéo dãn thời gian (nói chậm hơn ~20%): cost vẫn nhỏ
Xs = Xt[np.clip(np.round(np.arange(0, len(Xt), 0.8)).astype(int), 0, len(Xt) - 1)]
print(f"DTW_norm(X, X kéo dãn 1.25x) = {dtw(Xt, Xs)[0]:.3f}")

In [ ]:
def plot_dtw(X, Y, title, fname, ylab, xlab):
    dn, path, C, Dacc = dtw(X, Y)
    p = np.array(path)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    im = ax[0].imshow(C, origin="lower", aspect="auto", cmap="Blues")
    ax[0].plot(p[:, 1], p[:, 0], color="#d55e00", lw=1.6, label="optimal path")
    ax[0].plot([0, C.shape[1]-1], [0, C.shape[0]-1], "k--", lw=0.8, alpha=0.6, label="đường chéo")
    ax[0].set_title(f"Local-distance C[i,j] - {title}", fontsize=10); ax[0].grid(False); ax[1].grid(False)
    ax[0].set_xlabel(f"Frame j của {xlab}"); ax[0].set_ylabel(f"Frame i của {ylab}"); ax[0].legend(fontsize=8)
    fig.colorbar(im, ax=ax[0], label="Khoảng cách Euclid")
    im2 = ax[1].imshow(Dacc, origin="lower", aspect="auto", cmap="viridis")
    ax[1].plot(p[:, 1], p[:, 0], color="w", lw=1.2)
    ax[1].set_title(f"Ma trận tích luỹ D - DTW_norm = {dn:.3f}, |P| = {len(path)}", fontsize=10)
    ax[1].set_xlabel(f"Frame j của {xlab}"); ax[1].set_ylabel(f"Frame i của {ylab}")
    fig.colorbar(im2, ax=ax[1], label="Chi phí tích luỹ")
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, bbox_inches="tight"); plt.show()
    # độ lệch khỏi đường chéo (chuẩn hoá về [0,1] theo 2 trục)
    dev = np.mean(np.abs(p[:, 0] / max(1, C.shape[0]-1) - p[:, 1] / max(1, C.shape[1]-1)))
    n_h = int(np.sum(np.diff(p[:, 0]) == 0)); n_v = int(np.sum(np.diff(p[:, 1]) == 0))
    return dn, dev, n_h, n_v, C.shape

feat = lambda f: mfcc_feature(endpoint_detect(load_audio(f))[0])
wA, wB = LABELS[0], LABELS[1]
fa1, fa2, fb1 = train_files[wA][0], train_files[wA][1], train_files[wB][0]
res_same = plot_dtw(feat(fa1), feat(fa2), f"cùng từ '{VN[wA]}' ({fa1.name} vs {fa2.name})",
                    "E_dtw_same_word.png", fa1.name, fa2.name)
res_diff = plot_dtw(feat(fa1), feat(fb1), f"khác từ '{VN[wA]}' vs '{VN[wB]}'",
                    "E_dtw_diff_word.png", fa1.name, fb1.name)
cmp = pd.DataFrame([res_same, res_diff], index=["cùng từ", "khác từ"],
                   columns=["DTW_norm", "lệch chéo TB", "#bước ngang", "#bước dọc", "kích thước C (N×M)"])
display(cmp)
print(f"DTW_norm khác từ / cùng từ = {res_diff[0] / res_same[0]:.2f} lần")

In [ ]:
# Ma trận DTW_norm trung bình giữa các lớp (trên tập train) - kiểm tra tổng quát hơn 1 cặp
train_feats = {lab: [feat(f) for f in train_files[lab]] for lab in LABELS}
K = len(LABELS); Mavg = np.zeros((K, K))
for a, la in enumerate(LABELS):
    for b, lb in enumerate(LABELS):
        ds = [dtw(X, Y)[0] for ia, X in enumerate(train_feats[la]) for ib, Y in enumerate(train_feats[lb])
              if not (la == lb and ia >= ib)]
        Mavg[a, b] = np.mean(ds)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(Mavg, cmap="Blues")
for a in range(K):
    for b in range(K):
        ax.text(b, a, f"{Mavg[a,b]:.2f}", ha="center", va="center", fontsize=9,
                color="w" if Mavg[a, b] > Mavg.mean() else "k")
ax.set_xticks(range(K)); ax.set_xticklabels([VN[l] for l in LABELS])
ax.set_yticks(range(K)); ax.set_yticklabels([VN[l] for l in LABELS]); ax.grid(False)
ax.set_title("DTW_norm trung bình giữa các lớp (train)", fontsize=10); fig.colorbar(im)
fig.tight_layout(); fig.savefig(FIG_DIR / "E_dtw_class_matrix.png", bbox_inches="tight"); plt.show()
print(f"TB trong lớp (đường chéo) = {np.mean(np.diag(Mavg)):.3f} | "
      f"TB giữa các lớp = {Mavg[~np.eye(K, dtype=bool)].mean():.3f}")

**Nhận xét (E):**
- **Cùng từ:** optimal path bám sát đường chéo (độ lệch trung bình nhỏ), chỉ có vài đoạn ngang/dọc ngắn do một lần nói kéo dài hoặc nén một âm (thường là nguyên âm hoặc âm mũi cuối). Các ô trên đường đi có màu nhạt (local distance nhỏ) → DTW_norm thấp.
- **Khác từ:** không có "thung lũng" chi phí thấp liền mạch dọc đường chéo; path phải đi qua các ô đậm và thường có các đoạn ngang/dọc dài hơn → DTW_norm **cao hơn rõ rệt** (tỉ số in ở trên). Ma trận DTW trung bình giữa các lớp cho thấy đường chéo (cùng lớp) nhỏ hơn các ô ngoài chéo — đây là cơ sở để luật nearest-template hoạt động.
- Kiểm tra bắt buộc đạt: DTW_norm(X, X) = 0 và path đúng đường chéo; khi kéo dãn thời gian 1.25 lần, cost vẫn nhỏ hơn nhiều so với hai từ khác nhau.

---
## F. Bộ nhận dạng nearest-template

$D_w(X) = \min_r \mathrm{DTW_{norm}}(X, T_{w,r})$, $\hat w = \arg\min_w D_w(X)$.

**Reject (tuỳ chọn):** nếu $\min_w D_w(X) > \theta$ thì trả về `'unknown'`. Cách chọn θ **chỉ dùng tập train** (không nhìn test): với mỗi template, tính DTW_norm tới template *cùng lớp* gần nhất (leave-one-out) → lấy θ = max của các giá trị đó × 1.2 (biên an toàn 20%).

In [ ]:
FEATURE_FN = mfcc_feature        # đặc trưng mặc định: 13 MFCC + CMN

def extract(path, trim=True, feature_fn=None):
    y = load_audio(path)
    if trim:
        y, _ = endpoint_detect(y)
    return (feature_fn or FEATURE_FN)(y)

def build_templates(files_dict, trim=True, feature_fn=None, n_per_word=None):
    return {lab: [extract(f, trim, feature_fn) for f in fs[:n_per_word]] for lab, fs in files_dict.items()}

def recognize(path, templates, trim=True, feature_fn=None, theta=None):
    X = extract(path, trim, feature_fn)
    scores = {lab: min(dtw(X, R)[0] for R in refs) for lab, refs in templates.items()}
    ranked = dict(sorted(scores.items(), key=lambda kv: kv[1]))
    pred = next(iter(ranked))
    if theta is not None and ranked[pred] > theta:
        pred = "unknown"
    return pred, ranked

t0 = time.time()
templates = build_templates(train_files)
print(f"Đã tạo templates: { {VN[k]: len(v) for k, v in templates.items()} } trong {time.time()-t0:.1f} s")

# Chọn θ từ tập train (leave-one-out trong cùng lớp)
intra = []
for lab, refs in templates.items():
    for i, X in enumerate(refs):
        intra.append(min(dtw(X, R)[0] for k, R in enumerate(refs) if k != i))
THETA = 1.2 * max(intra)
print(f"DTW_norm trong lớp (train, LOO): min={min(intra):.3f}, TB={np.mean(intra):.3f}, max={max(intra):.3f}"
      f" -> θ = 1.2 × max = {THETA:.3f}")

In [ ]:
# Nhận dạng toàn bộ tập test + bảng top-3
rows = []
t0 = time.time()
for lab in LABELS:
    for f in test_files[lab]:
        pred, ranked = recognize(f, templates)
        top = list(ranked.items())
        rows.append(dict(test_file=f"{lab}/{f.name}", true_label=lab, pred_label=pred,
                         pred_with_reject="unknown" if top[0][1] > THETA else pred,
                         top1_label=top[0][0], top1_dtw=round(top[0][1], 4),
                         top2_label=top[1][0], top2_dtw=round(top[1][1], 4),
                         top3_label=top[2][0] if len(top) > 2 else "", top3_dtw=round(top[2][1], 4) if len(top) > 2 else np.nan,
                         margin_top2_top1=round(top[1][1] - top[0][1], 4),
                         correct=pred == lab))
res = pd.DataFrame(rows)
print(f"Nhận dạng {len(res)} file test trong {time.time()-t0:.1f} s")
display(res[["test_file", "true_label", "pred_label", "top1_label", "top1_dtw", "top2_label", "top2_dtw",
             "top3_label", "top3_dtw", "correct"]])

res[["test_file", "true_label", "pred_label", "top1_label", "top1_dtw", "top2_label", "top2_dtw",
     "pred_with_reject", "correct"]].to_csv("results.csv", index=False, encoding="utf-8")
print("Đã lưu results.csv")

### F.2. Demo: nhận dạng một file WAV chưa biết
Hàm `recognize_file` là **kết quả đầu ra bắt buộc**: nhận đường dẫn WAV bất kỳ → trim → MFCC → DTW với toàn bộ template → in nhãn dự đoán và khoảng cách DTW.
Đặt `DEMO_RECORD = True` để nói một từ mới vào micro rồi nhận dạng ngay.

In [ ]:
def recognize_file(path, theta=THETA):
    pred, ranked = recognize(Path(path), templates, theta=theta)
    print(f"File: {path}")
    print(f"  => Dự đoán: '{VN.get(pred, pred)}' ({pred})")
    for k, (lab, d) in enumerate(ranked.items(), 1):
        print(f"     top-{k}: {VN.get(lab, lab):<6} DTW_norm = {d:.4f}")
    return pred, ranked

DEMO_RECORD = False
DEMO_FILE = Path("demo") / "unknown.wav"
if DEMO_RECORD:
    input("Nhấn Enter rồi nói một từ trong vocabulary...")
    record_utterance(DEMO_FILE)
elif not DEMO_FILE.exists():
    DEMO_FILE = test_files[LABELS[-1]][0]          # dùng một file test nếu chưa ghi file demo
_ = recognize_file(DEMO_FILE)

---
## G. Đánh giá và thí nghiệm
### G.1. Accuracy và confusion matrix (baseline: trim + 13 MFCC + CMN, 3 template/từ)

In [ ]:
def plot_cm(y_true, y_pred, labels, title, fname, ax=None):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(cm, cmap="Blues", vmin=0)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="w" if cm[i, j] > cm.max() / 2 else "k")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels([VN.get(l, l) for l in labels], rotation=30)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels([VN.get(l, l) for l in labels])
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Nhãn thật"); ax.set_title(title, fontsize=10); ax.grid(False)
    if own:
        plt.colorbar(im, ax=ax); plt.tight_layout(); plt.savefig(FIG_DIR / fname, bbox_inches="tight"); plt.show()
    return cm

acc_base = accuracy_score(res.true_label, res.pred_label)
print(f"Accuracy baseline = {acc_base*100:.1f}% ({res.correct.sum()}/{len(res)})")
cm_base = plot_cm(res.true_label, res.pred_label, LABELS,
                  f"Confusion matrix - baseline (acc = {acc_base*100:.1f}%)", "G_confusion_matrix.png")
n_rej = (res.pred_with_reject == "unknown").sum()
print(f"Với reject θ = {THETA:.3f}: {n_rej} file bị trả về 'unknown'")

# Cặp từ dễ nhầm nhất: dựa vào ô ngoài chéo lớn nhất; nếu không có lỗi -> cặp có margin top2-top1 nhỏ nhất
off = cm_base.copy(); np.fill_diagonal(off, 0)
if off.max() > 0:
    i, j = np.unravel_index(np.argmax(off), off.shape)
    CONF_PAIR = (LABELS[i], LABELS[j]); why = f"{off[i, j]} lần '{VN[LABELS[i]]}' bị nhận thành '{VN[LABELS[j]]}'"
else:
    r = res.sort_values("margin_top2_top1").iloc[0]
    CONF_PAIR = (r.true_label, r.top2_label); why = f"không có lỗi; margin top2-top1 nhỏ nhất = {r.margin_top2_top1:.3f} ({r.test_file})"
print(f"Cặp dễ nhầm nhất: '{VN[CONF_PAIR[0]]}' ↔ '{VN[CONF_PAIR[1]]}' ({why})")

### G.2. Thí nghiệm có kiểm soát (E1–E4)
Mỗi thí nghiệm **chỉ thay đổi một yếu tố**, giữ nguyên tập train/test:

| Mã | Thay đổi duy nhất | Giữ cố định |
|---|---|---|
| E1 | Không trim vs. có trim | MFCC/DTW/templates |
| E2 | 13 MFCC vs. 13 MFCC + Δ (26 chiều) | Train/test split, trim |
| E3 | 1 template/từ vs. 3 templates/từ | Feature, test set |
| E4 | Same-speaker vs. cross-speaker (chỉ chạy khi có `dataset_speaker2/`) | Vocabulary/feature |

In [ ]:
def evaluate(trim=True, feature_fn=None, n_per_word=None, test_set=None, tmpl_files=None):
    tm = build_templates(tmpl_files or train_files, trim=trim, feature_fn=feature_fn, n_per_word=n_per_word)
    yt, yp, best, gap = [], [], [], []
    for lab, fs in (test_set or test_files).items():
        for f in fs:
            pred, ranked = recognize(f, tm, trim=trim, feature_fn=feature_fn)
            v = list(ranked.values())
            yt.append(lab); yp.append(pred); best.append(ranked[lab]); gap.append(v[1] - v[0])
    return dict(acc=accuracy_score(yt, yp), y_true=yt, y_pred=yp,
                mean_dtw_true=np.mean(best), mean_gap=np.mean(gap))

experiments = {}
t0 = time.time()
experiments["Baseline (trim, MFCC13, 3 tmpl)"] = evaluate()
experiments["E1: không trim"]                  = evaluate(trim=False)
experiments["E2: MFCC13 + Δ"]                  = evaluate(feature_fn=mfcc_delta)
experiments["E3: 1 template/từ"]               = evaluate(n_per_word=1)
if DATA_DIR2.exists() and list_files(DATA_DIR2):
    spk2 = {l: v for l, v in list_files(DATA_DIR2).items() if l in LABELS and v}
    experiments["E4: cross-speaker (test = người 2)"] = evaluate(test_set=spk2)
else:
    print("Không có dataset_speaker2/ -> bỏ qua E4 (cross-speaker).")
print(f"Chạy xong {len(experiments)} cấu hình trong {time.time()-t0:.1f} s")

summary = pd.DataFrame({k: dict(accuracy_pct=round(100*v["acc"], 1),
                                n_correct=f"{sum(a == b for a, b in zip(v['y_true'], v['y_pred']))}/{len(v['y_true'])}",
                                mean_DTW_to_true_class=round(v["mean_dtw_true"], 3),
                                mean_margin_top2_top1=round(v["mean_gap"], 3))
                        for k, v in experiments.items()}).T
display(summary)
summary.to_csv(FIG_DIR / "G_experiments_summary.csv", encoding="utf-8")

In [ ]:
names = list(experiments)
fig, axes = plt.subplots(1, len(names), figsize=(4.3*len(names), 4))
for ax, n in zip(np.atleast_1d(axes), names):
    v = experiments[n]
    plot_cm(v["y_true"], v["y_pred"], LABELS, f"{n}\nacc = {100*v['acc']:.1f}%", None, ax=ax)
fig.tight_layout(); fig.savefig(FIG_DIR / "G_experiments_confusion.png", bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].barh(names, summary.accuracy_pct.astype(float), color="#4e79a7"); ax[0].set_xlabel("Accuracy (%)"); ax[0].set_xlim(0, 105)
ax[1].barh(names, summary.mean_margin_top2_top1.astype(float), color="#f28e2b"); ax[1].set_xlabel("Margin TB (DTW top-2 − top-1)")
for a in ax: a.invert_yaxis()
fig.suptitle("G. So sánh các thí nghiệm"); fig.tight_layout()
fig.savefig(FIG_DIR / "G_experiments_bar.png", bbox_inches="tight"); plt.show()

In [ ]:
# Nhận xét tự động dựa trên số liệu thực tế của các thí nghiệm
b = experiments["Baseline (trim, MFCC13, 3 tmpl)"]
def _cmp(name, if_better, if_worse):
    e = experiments[name]
    d_acc = 100 * (e["acc"] - b["acc"]); d_gap = e["mean_gap"] - b["mean_gap"]
    trend = "tăng" if d_acc > 0 else ("giảm" if d_acc < 0 else "không đổi")
    verdict = if_better if (d_acc > 0 or (d_acc == 0 and d_gap > 0)) else if_worse
    print(f"- {name}: accuracy {100*e['acc']:.1f}% vs baseline {100*b['acc']:.1f}% ({trend} {abs(d_acc):.1f} điểm); "
          f"margin top2−top1 TB {e['mean_gap']:.3f} vs {b['mean_gap']:.3f} ({'+' if d_gap >= 0 else ''}{d_gap:.3f}). {verdict}")

_cmp("E1: không trim",
     "Trên dữ liệu này bỏ trim không làm xấu đi - có thể silence đầu/cuối ngắn và đồng đều giữa các file; cần kiểm tra lại trên file có silence dài.",
     "Bỏ trim làm các lớp khó tách hơn: DTW phải căn cả các đoạn silence giống nhau ở mọi từ, kéo khoảng cách giữa các lớp lại gần nhau.")
_cmp("E2: MFCC13 + Δ",
     "Δ bổ sung thông tin chuyển động phổ (chuyển tiếp phụ âm-nguyên âm, formant) nên tách lớp tốt hơn.",
     "Δ không giúp trên dữ liệu này: Δ nhạy nhiễu và tăng số chiều; với ít dữ liệu, phần tĩnh đã đủ phân biệt.")
_cmp("E3: 1 template/từ",
     "1 template vẫn đủ tốt trên dữ liệu này - các lần nói khá đồng đều.",
     "Chỉ 1 template không bao phủ được biến thiên giữa các lần nói; 3 template ổn định hơn.")
if "E4: cross-speaker (test = người 2)" in experiments:
    e4 = experiments["E4: cross-speaker (test = người 2)"]
    print(f"- E4: DTW TB tới lớp đúng {e4['mean_dtw_true']:.3f} vs same-speaker {b['mean_dtw_true']:.3f} "
          f"(tăng {100*(e4['mean_dtw_true']/b['mean_dtw_true']-1):.0f}%).")

### G.3. Phân tích cặp từ dễ nhầm nhất
Cell dưới tự chọn cặp dễ nhầm nhất (ô ngoài chéo lớn nhất của confusion matrix; nếu không có lỗi thì chọn cặp có margin top-2/top-1 nhỏ nhất) và vẽ waveform, MFCC và DTW path để phân tích nguyên nhân.

In [ ]:
wa, wb = CONF_PAIR
fa, fb = test_files[wa][0], train_files[wb][0]
Xa, Xb = feat(fa), feat(fb)
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for k, (w, f, X) in enumerate([(wa, fa, Xa), (wb, fb, Xb)]):
    yt, _ = endpoint_detect(load_audio(f))
    axes[0, k].plot(np.arange(len(yt))/FS, yt, lw=0.5); axes[0, k].set_title(f"'{VN[w]}' ({f.name}) - đã trim", fontsize=10)
    axes[0, k].set_xlabel("Thời gian (s)")
    vl = np.percentile(np.abs(X[:, 1:]), 98)
    axes[1, k].imshow(X.T, origin="lower", aspect="auto", cmap="coolwarm", vmin=-vl, vmax=vl); axes[1, k].grid(False)
    axes[1, k].set_title(f"MFCC '{VN[w]}' {X.shape}", fontsize=10); axes[1, k].set_xlabel("Frame")
fig.tight_layout(); fig.savefig(FIG_DIR / "G_confused_pair.png", bbox_inches="tight"); plt.show()
r = plot_dtw(Xa, Xb, f"cặp dễ nhầm '{VN[wa]}' vs '{VN[wb]}'", "G_confused_pair_dtw.png", fa.name, fb.name)
print(f"DTW_norm('{VN[wa]}' test, '{VN[wb]}' template) = {r[0]:.3f}; "
      f"DTW trung bình trong lớp '{VN[wa]}' = {Mavg[LABELS.index(wa), LABELS.index(wa)]:.3f}")

---
## Báo cáo

### 1. Bảng tham số đã dùng
| Tham số | Giá trị | Ghi chú |
|---|---|---|
| $F_s$ | 16 000 Hz | mọi file được resample/đưa về mono 16 kHz PCM-16 |
| Frame / Hop | 25 ms (400 mẫu) / 10 ms (160 mẫu) | 100 frame/s, chồng lấn 15 ms |
| Cửa sổ | Hamming (tự cài) | ZCR dùng cửa sổ chữ nhật |
| Endpoint | `TOP_DB` = 35 dB dưới đỉnh, sàn = nền + 10 dB; $T_Z = \max(0.25, \mu_Z + 2\sigma_Z)$, tìm trong 250 ms; margin 50 ms | heuristic của lab |
| Pre-emphasis | α = 0.97 | dùng cho cả train và test |
| NFFT | 512 | |
| Mel filters | 24, thang $1125\ln(1+f/700)$, 0–8 kHz | |
| MFCC | 13 hệ số ($c_0..c_{12}$), DCT theo công thức đề, **CMN** theo utterance | E2: + Δ (hồi quy ±2 frame) → 26 chiều |
| Local distance | Euclid | |
| DTW | 3 bước (ngang/dọc/chéo), chuẩn hoá theo $|P|$ | tự cài bằng quy hoạch động |
| Quyết định | nearest-template, min theo 3 template/từ | reject θ = 1.2 × max DTW trong lớp (LOO, train) |

### 2. Kết quả thí nghiệm E1–E2 (và E3/E4)
Xem bảng `summary` và nhận xét tự động ở mục G.2 (lưu ở `figures/G_experiments_summary.csv`). Kỳ vọng lý thuyết:
- **E1:** bỏ endpoint detection làm các đoạn silence đầu/cuối được căn chỉnh với nhau; silence của mọi từ giống nhau nên kéo các khoảng cách giữa lớp lại gần nhau (margin giảm), và nếu độ dài silence khác nhau nhiều, path bị lệch khỏi đường chéo ở hai đầu → accuracy giảm hoặc giữ nguyên nhưng kém ổn định.
- **E2:** Δ mô tả tốc độ thay đổi phổ (chuyển tiếp phụ âm–nguyên âm, chuyển động formant trong "hai", "bốn"), bổ sung cho MFCC tĩnh → thường giảm nhầm lẫn giữa các từ có cùng nguyên âm chính.
- **E3:** 3 template/từ giúp ổn định hơn 1 template vì bao phủ biến thiên tốc độ/âm lượng giữa các lần nói.

### 3. Trả lời câu hỏi báo cáo
**Câu 1. Vì sao không nên dùng toàn bộ waveform làm template?**
Waveform rất nhạy với pha, biên độ, nhiễu và độ trễ: hai lần nói cùng một từ có thể có dạng sóng tương quan rất thấp dù nghe giống nhau. Khi thời lượng khác nhau, phép so sánh từng mẫu không có nghĩa và chi phí DTW trên hàng chục nghìn mẫu là cực lớn ($O(NM)$ với $N, M \sim 10^4$). Đặc trưng ngắn hạn như MFCC nén mỗi 10 ms thành 13 số mô tả **đường bao phổ** — phần mang thông tin âm vị — và bỏ qua pha, F0 chi tiết, nên vừa gọn vừa bền vững hơn.

**Câu 2. Vai trò của short-time energy và ZCR trong endpoint detection?**
Energy tách nhanh vùng có tiếng nói khỏi silence vì nguyên âm/âm hữu thanh có năng lượng cao hơn nền hàng chục dB — đó là bước tìm vùng speech thô. Nhưng phụ âm vô thanh (/x/ trong "không", /h/ trong "hai", /s/, /f/) có năng lượng thấp, gần mức nền, nên chỉ dùng energy sẽ cắt mất. ZCR của các phụ âm này cao (phổ dạng nhiễu tần số cao) trong khi silence sạch có ZCR thấp, nên ZCR dùng để **tinh chỉnh biên** — kéo điểm bắt đầu/kết thúc ra ngoài để giữ phụ âm.

**Câu 3. Vì sao Mel filterbank có khoảng cách theo Hz rộng dần khi tần số tăng?**
Thang Mel mô phỏng cảm nhận cao độ của tai: gần tuyến tính dưới ~1 kHz và gần logarit ở tần số cao (tai phân biệt tốt 100 Hz vs 200 Hz nhưng khó phân biệt 6000 vs 6100 Hz). Các bộ lọc cách **đều trên thang Mel**; ánh xạ ngược $f = 700(e^{b/1125}-1)$ là hàm mũ nên các bước Mel bằng nhau tương ứng các khoảng Hz ngày càng rộng. Kết quả là độ phân giải cao ở vùng formant F1/F2 quan trọng, thấp ở vùng cao tần.

**Câu 4. Log trong MFCC tác dụng gì? DCT biến M log-energy thành gì?**
Log nén dynamic range (năng lượng các băng chênh nhau nhiều bậc độ lớn → vài chục đơn vị), gần với cảm nhận độ to (dB), và biến **tích** nguồn × kênh/tuyến âm thành **tổng**, nhờ đó ảnh hưởng của micro/kênh truyền trở thành một hằng số cộng có thể trừ bằng CMN. DCT biến 24 log-energy (tương quan mạnh giữa các băng kề nhau) thành các **hệ số cepstral** gần như không tương quan: $c_0$ ~ log năng lượng tổng, các hệ số bậc thấp mô tả hình dạng đường bao phổ (độ nghiêng, vị trí formant), hệ số bậc cao mô tả chi tiết nhanh (hài của F0). Giữ 13 hệ số đầu = làm trơn phổ, giữ thông tin tuyến âm và loại bỏ phần lớn thông tin pitch.

**Câu 5. Ý nghĩa bước ngang, bước dọc, bước chéo trong ma trận DTW?**
Với $i$ là frame của X (trục dọc), $j$ là frame của Y (trục ngang): **bước chéo** $(i-1,j-1)\to(i,j)$ — hai chuỗi cùng tiến 1 frame, tốc độ như nhau; **bước dọc** $(i-1,j)\to(i,j)$ — X tiến còn Y đứng yên, tức một frame của Y được ghép với nhiều frame của X (X nói chậm/kéo dài đoạn đó hơn); **bước ngang** $(i,j-1)\to(i,j)$ — ngược lại, Y kéo dài hơn X. Path toàn chéo nghĩa là hai lần nói cùng nhịp độ.

**Câu 6. Tại sao phải chuẩn hoá DTW cost theo path length?**
$D[N-1,M-1]$ là **tổng** local distance dọc đường đi; utterance dài hơn có path dài hơn nên tổng lớn hơn dù từng cặp frame khớp tốt. Nếu không chuẩn hoá, template dài bị bất lợi và template ngắn được ưu tiên một cách sai lệch. Chia cho $|P|$ cho ra **khoảng cách trung bình mỗi cặp frame**, so sánh công bằng giữa các template có độ dài khác nhau.

**Câu 7. Ba nguyên nhân làm cùng một từ có MFCC khác nhau giữa hai lần nói:**
(1) **Tốc độ nói và nhịp điệu** khác nhau → số frame và độ dài từng âm khác nhau; (2) **âm lượng, khoảng cách/hướng micro, đặc tính kênh** → thay đổi $c_0$ và độ nghiêng phổ (CMN chỉ bù được phần tĩnh); (3) **nhiễu nền và lỗi endpoint** (cắt thiếu/thừa) → thêm các frame không thuộc từ; ngoài ra còn **ngữ điệu/thanh điệu, đồng phát âm (coarticulation), trạng thái người nói** (mệt, cảm) làm dịch formant.

**Câu 8. Cặp từ dễ nhầm nhất và nguyên nhân:** cặp được xác định tự động ở cuối mục G.1 và phân tích bằng hình ở G.3 (`figures/G_confused_pair*.png`). Hướng phân tích: (i) các từ có **cùng nguyên âm chính** (ví dụ "ba" và "hai" đều chứa /a/ dài; "một" và "không" đều có nguyên âm tròn môi /o/) có vùng MFCC giữa từ gần như giống nhau, nên DTW path đi qua vùng chi phí thấp dài; (ii) phân biệt chủ yếu nằm ở **phụ âm đầu ngắn** (/b/, /h/, /m/) chỉ chiếm vài frame nên đóng góp ít vào chi phí trung bình; (iii) **thanh điệu** (sắc, nặng, ngang) chủ yếu nằm ở F0, mà 13 MFCC gần như loại bỏ F0 — nên khác biệt thanh không giúp tách lớp; (iv) từ rất ngắn như "một" (âm tắc cuối /t/ và thanh nặng) dễ bị trim sai. Biện pháp: thêm Δ/ΔΔ, tăng số template, cải thiện endpoint.

**Câu 9. Hạn chế của DTW khi nhận dạng người nói mới chưa có template; Chương 3 giải quyết thế nào?**
DTW so khớp với **các mẫu cụ thể** của người đã ghi: không có mô hình thống kê về biến thiên, nên với người nói mới (giọng, giới tính, phương ngữ khác) khoảng cách tới mọi template đều tăng và dễ nhầm; muốn mở rộng phải ghi template cho từng người/từng từ, chi phí tính toán tăng tuyến tính theo số template, và không thể nhận dạng từ ngoài vocabulary hay câu liên tục. Chương 3 dùng **mô hình thống kê HMM (với GMM/mạng nơ-ron làm mô hình phát xạ)** huấn luyện trên nhiều người nói, mô hình hoá đơn vị âm vị cùng **từ điển phát âm** và **mô hình ngôn ngữ**, nên tổng quát hoá cho người nói mới và vocabulary lớn hơn.

### 4. Kết luận
- Đã xây dựng hoàn chỉnh hệ nhận dạng từ đơn speaker-dependent: đọc WAV → endpoint detection (energy + ZCR) → MFCC tự cài (pre-emphasis, Hamming, FFT, Mel 24, log, DCT 13, CMN) → DTW tự cài (DP + backtracking + chuẩn hoá $|P|$) → nearest-template với tuỳ chọn reject.
- DTW_norm cùng từ nhỏ hơn rõ rệt so với khác từ, path cùng từ bám đường chéo — xác nhận DTW + MFCC phù hợp cho vocabulary nhỏ.
- Các thí nghiệm có kiểm soát (E1–E3, E4 nếu có dữ liệu người 2) định lượng vai trò của endpoint detection, đặc trưng động Δ và số template; kết quả cụ thể ở bảng G.2.
- Hạn chế: phụ thuộc người nói, nhạy với endpoint và điều kiện thu; hướng mở rộng là HMM/mô hình thống kê ở Chương 3.

**Sản phẩm nộp:** `lab02_2351260684.ipynb`, `dataset/`, `trimmed/`, `figures/` (waveform, energy/ZCR, endpoint, Mel filterbank, MFCC heatmap, DTW path, confusion matrix), `results.csv`.